# Synthetic CDV Experiment - Heterogeneity Sweep

This notebook runs the full multi-seed experiment comparing Global vs CDV methods
across 4 levels of causal heterogeneity (alpha).

**Protocol:**
- For each alpha in {0.0, 0.33, 0.66, 1.0}:
  - Fix test/val features (same cases across all seeds)
  - For each seed in {0, ..., 499}:
    - Generate fresh training data
    - Train global model vs CDV model
    - Evaluate on fixed test set with known ground-truth CATE
  - Save results per alpha

In [ ]:
import os
import json as json_lib
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

from cdv_utils.synthetic_dgp import (
    generate_synthetic_dataset,
    generate_counterfactuals_for_fixed_features,
    get_w_cols,
    MISSING_VALUE
)
from cdv_utils.causal_modeling import (
    assign_variants_by_patterns,
    group_by_variants_with_filtered_columns,
    process_test_data_with_training_variants
)
from cdv_utils.experiment_runner import (
    run_multi_seed_synthetic_experiment,
    load_experiment_results
)

# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================
ALPHA_VALUES = [0.0, 0.33, 0.66, 1.0]   # Heterogeneity levels
N_TRAIN = 3000                           # Training samples per seed
N_TEST = 1000                            # Fixed test set size
N_VAL = 1000                             # Fixed validation set size
NUM_SEEDS = 500                          # Monte Carlo replications

# Variant configuration:
# NUM_TOP_VARIANTS = number of modeled top patterns (user selects from elbow chart)
# K = total variants including the "others" bucket = NUM_TOP_VARIANTS + 1
# Top variants get their own variant-specific model.
# The "others" variant (K-th) uses the global model (all features, all data).
NUM_TOP_VARIANTS = 3                     # Number of top variant patterns to model
K = NUM_TOP_VARIANTS + 1                 # Total variants (top patterns + 1 others bucket)

R2_THRESHOLD = 0.1                       # Model selection threshold
INITIAL_SEED = 420                       # Estimator initialization seed
TEST_VAL_SEED = 999                      # Fixed seed for test/val features

w_cols = get_w_cols()
experiment_seeds = list(range(NUM_SEEDS))

print("Experiment Configuration:")
print(f"  Alpha values: {ALPHA_VALUES}")
print(f"  Training size per seed: {N_TRAIN}")
print(f"  Test size: {N_TEST}")
print(f"  Validation size: {N_VAL}")
print(f"  Number of seeds: {NUM_SEEDS}")
print(f"  Top variant patterns modeled: {NUM_TOP_VARIANTS}")
print(f"  Total variants (incl. others): {K}")
print(f"  Variant-specific models: {NUM_TOP_VARIANTS} (top variants)")
print(f"  Others variant (variant {K}): uses global model")
print(f"  Feature columns: {w_cols}")
print(f"\nEnvironment setup complete!")

## 2. Generate Fixed Test and Validation Features

Features remain constant across all alpha values and seeds.
Only treatment/outcomes are regenerated per alpha.

In [ ]:
# Generate fixed test and validation features
df_test_full = generate_synthetic_dataset(n=N_TEST, alpha=0.5, seed=TEST_VAL_SEED)
df_val_full = generate_synthetic_dataset(n=N_VAL, alpha=0.5, seed=TEST_VAL_SEED + 1)

# Extract fixed features (these never change)
feature_cols_plus_sg = w_cols + ["subgroup"]
df_test_features = df_test_full[feature_cols_plus_sg].copy()
df_val_features = df_val_full[feature_cols_plus_sg].copy()

print(f"Test features shape: {df_test_features.shape}")
print(f"Validation features shape: {df_val_features.shape}")
print(f"\nTest sub-group distribution:")
print(df_test_features["subgroup"].value_counts().sort_index())


In [ ]:
# Discover top variant patterns from the test features
feature_mask = df_test_features[w_cols] >= 0
test_patterns = feature_mask.apply(lambda row: "".join(["1" if val else "0" for val in row]), axis=1)

# Get top NUM_TOP_VARIANTS patterns (determined from elbow chart in NB04)
pattern_counts = test_patterns.value_counts()
top_variants = list(pattern_counts.index[:NUM_TOP_VARIANTS])

print("Discovered variant patterns:")
print("=" * 60)
for i, pattern in enumerate(top_variants, 1):
    present = [w_cols[j] for j, bit in enumerate(pattern) if bit == "1"]
    count = pattern_counts[pattern]
    pct = count / len(test_patterns) * 100
    print(f"  Variant {i}: pattern={pattern} | features={present} | n={count} ({pct:.1f}%)")

# Others bucket: all remaining patterns grouped into variant K
others_count = sum(pattern_counts.values[NUM_TOP_VARIANTS:])
others_pct = others_count / len(test_patterns) * 100
print(f"  Variant {K} (others): n={others_count} ({others_pct:.1f}%)")

top_coverage = sum(pattern_counts.values[:NUM_TOP_VARIANTS]) / len(test_patterns) * 100
print(f"\nTop {NUM_TOP_VARIANTS} coverage: {top_coverage:.1f}% (target: >85%)")
print(f"\nTotal models to fit: {K} variant-specific + 1 global")

# Build training_variant_patterns dict
training_variant_patterns = {i+1: pattern for i, pattern in enumerate(top_variants)}
print(f"\nTraining variant patterns: {training_variant_patterns}")

## 3. Prepare Variant-Specific DataFrames

In [ ]:
def prepare_test_val_for_alpha(df_features, alpha, seed, w_cols, top_variants, k, training_variant_patterns):
    """
    Given fixed features, generate counterfactuals for a specific alpha
    and create variant-specific DataFrames.
    """
    # Generate counterfactuals for this alpha
    df_full = generate_counterfactuals_for_fixed_features(df_features, alpha=alpha, seed=seed)
    
    # Keep only needed columns
    df_for_variants = df_full[w_cols + ["t", "y", "y0", "y1", "ite"]].copy()
    
    # Assign variants
    df_for_variants = assign_variants_by_patterns(df_for_variants, top_variants, k)
    
    # Create variant-specific DFs (CDV method)
    variant_dfs, variant_info = group_by_variants_with_filtered_columns(df_for_variants, num_variants=k)
    
    # Create global variant DFs (all features for each variant test cases)
    global_variant_dfs = {}
    for variant_num in variant_dfs.keys():
        variant_rows = df_for_variants[df_for_variants["variant"] == variant_num].copy()
        essential_cols = ["t", "y", "y0", "y1", "ite"]
        global_variant_dfs[variant_num] = variant_rows[essential_cols + w_cols].copy()
    
    return variant_dfs, global_variant_dfs, variant_info


# Verify structure with reference alpha
test_variant_dfs, global_test_variant_dfs, test_info = prepare_test_val_for_alpha(
    df_test_features, alpha=0.5, seed=TEST_VAL_SEED,
    w_cols=w_cols, top_variants=top_variants, k=K,
    training_variant_patterns=training_variant_patterns
)

print("Test variant structure (reference alpha=0.5):")
print(test_info[["variant", "count", "percent", "pattern", "num_features"]].to_string(index=False))

print("\nVariant DataFrame columns:")
for v, vdf in test_variant_dfs.items():
    print(f"  Variant {v}: {list(vdf.columns)} (n={len(vdf)})")


## 4. Run Experiments Across Heterogeneity Levels

In [ ]:
# Main experiment loop
os.makedirs("results", exist_ok=True)

for alpha in ALPHA_VALUES:
    results_save_path = f"results/synthetic_alpha_{alpha:.2f}.pkl"
    
    # Check if already completed
    if os.path.exists(results_save_path):
        existing = load_experiment_results(results_save_path)
        if len(existing) >= NUM_SEEDS:
            print(f"\nAlpha={alpha}: Already completed ({len(existing)} seeds). Skipping.")
            continue
        else:
            print(f"\nAlpha={alpha}: Resuming from {len(existing)} seeds...")
    
    print(f"\n" + "=" * 70)
    print(f"RUNNING EXPERIMENT: alpha = {alpha}")
    print("=" * 70)
    
    # Regenerate test/val counterfactuals for this alpha
    test_variant_dfs_alpha, global_test_variant_dfs_alpha, _ = prepare_test_val_for_alpha(
        df_test_features, alpha=alpha, seed=TEST_VAL_SEED,
        w_cols=w_cols, top_variants=top_variants, k=K,
        training_variant_patterns=training_variant_patterns
    )
    
    val_variant_dfs_alpha, global_val_variant_dfs_alpha, _ = prepare_test_val_for_alpha(
        df_val_features, alpha=alpha, seed=TEST_VAL_SEED + 1,
        w_cols=w_cols, top_variants=top_variants, k=K,
        training_variant_patterns=training_variant_patterns
    )
    
    # Run multi-seed experiment for this alpha
    results_by_seed = run_multi_seed_synthetic_experiment(
        alpha=alpha,
        experiment_seeds=experiment_seeds,
        n_train=N_TRAIN,
        w_cols=w_cols,
        top_variants=top_variants,
        k=K,
        training_variant_patterns=training_variant_patterns,
        test_variant_dataframes=test_variant_dfs_alpha,
        val_variant_dataframes=val_variant_dfs_alpha,
        global_test_variant_dataframes=global_test_variant_dfs_alpha,
        global_val_variant_dataframes=global_val_variant_dfs_alpha,
        results_save_path=results_save_path,
        initial_seed=INITIAL_SEED,
        r2_threshold=R2_THRESHOLD
    )
    
    print(f"\nAlpha={alpha}: Completed with {len(results_by_seed)} seeds")

print("\n" + "=" * 70)
print("ALL EXPERIMENTS COMPLETE!")
print("=" * 70)


## 5. Results Checkpoint

In [ ]:
# Verify all results are saved
print("Results files:")
print("=" * 50)
for alpha in ALPHA_VALUES:
    results_path = f"results/synthetic_alpha_{alpha:.2f}.pkl"
    if os.path.exists(results_path):
        results = load_experiment_results(results_path)
        print(f"  alpha={alpha:.2f}: {len(results)} seeds completed")
    else:
        print(f"  alpha={alpha:.2f}: NOT FOUND")


In [ ]:
# Save experiment configuration
config = {
    "alpha_values": ALPHA_VALUES,
    "n_train": N_TRAIN,
    "n_test": N_TEST,
    "n_val": N_VAL,
    "num_seeds": NUM_SEEDS,
    "k": K,
    "r2_threshold": R2_THRESHOLD,
    "initial_seed": INITIAL_SEED,
    "test_val_seed": TEST_VAL_SEED,
    "w_cols": w_cols,
    "top_variants": top_variants,
    "training_variant_patterns": {str(k_v): v for k_v, v in training_variant_patterns.items()}
}

with open("results/synthetic_experiment_config.json", "w") as f:
    json_lib.dump(config, f, indent=2)

print("Experiment config saved to results/synthetic_experiment_config.json")
